In [8]:
# ライブラリのインポート
import collections
from ortools.sat.python import cp_model

In [9]:
import pandas as pd

m = int(input("機械数を入力してください"))
n = int(input("ジョブ数を入力してください"))
s = input('ファイル名を入力してください')

df = pd.read_csv(f'.\data\Benchmark\{s}.csv')
#df_t = pd.read_csv(f'/Users/kurozuhajime/Desktop/JSScheduling/data/Benchmark/opetime/{n}_{m}_py.csv')

ope_number = 0
jobs_data = []
for j in range(n):
    job = []
    for k in range(m):
        ope_number += 1
        idx_M = 'M_' + str(k+1)
        idx_T = 'P_' + str(k+1)
        kk = df[idx_M][j], df[idx_T][j]
        job.append(kk)
    jobs_data.append(job)
#jobs_data.append([])
print(ope_number)
print(jobs_data)


<string>:7: SyntaxWarning:

invalid escape sequence '\{'

<>:7: SyntaxWarning:

invalid escape sequence '\{'

<>:7: SyntaxWarning:

invalid escape sequence '\d'

<string>:7: SyntaxWarning:

invalid escape sequence '\{'

<>:7: SyntaxWarning:

invalid escape sequence '\{'

<>:7: SyntaxWarning:

invalid escape sequence '\d'

C:\Users\RJC238\AppData\Local\Temp\ipykernel_17144\2645082217.py:7: SyntaxWarning:

invalid escape sequence '\{'

C:\Users\RJC238\AppData\Local\Temp\ipykernel_17144\2645082217.py:7: SyntaxWarning:

invalid escape sequence '\d'



100
[[(1, 29), (2, 78), (3, 9), (4, 36), (5, 49), (6, 11), (7, 62), (8, 56), (9, 44), (10, 21)], [(1, 43), (3, 90), (5, 75), (10, 11), (4, 69), (2, 28), (7, 46), (6, 46), (8, 72), (9, 30)], [(2, 91), (1, 85), (4, 39), (3, 74), (9, 90), (6, 10), (8, 12), (7, 89), (10, 45), (5, 33)], [(2, 81), (3, 95), (1, 71), (5, 99), (7, 9), (9, 52), (8, 85), (4, 98), (10, 22), (6, 43)], [(3, 14), (1, 6), (2, 22), (6, 61), (4, 26), (5, 69), (9, 21), (8, 49), (10, 72), (7, 53)], [(3, 84), (2, 2), (6, 52), (4, 95), (9, 48), (10, 72), (1, 47), (7, 65), (5, 6), (8, 25)], [(2, 46), (1, 37), (4, 61), (3, 13), (7, 32), (6, 21), (10, 32), (9, 89), (8, 30), (5, 55)], [(3, 31), (1, 86), (2, 46), (6, 74), (5, 32), (7, 88), (9, 19), (10, 48), (8, 36), (4, 79)], [(1, 76), (2, 69), (4, 76), (6, 51), (3, 85), (10, 11), (7, 40), (8, 89), (5, 26), (9, 74)], [(2, 85), (1, 13), (3, 61), (7, 7), (9, 64), (10, 76), (6, 47), (4, 52), (5, 90), (8, 45)]]


In [10]:
import pandas as pd

# Data.

"""   
jobs_data = [  # task = (machine_id, processing_time).
    [(0, 3), (1, 2), (2, 2)],  # Job0
    [(0, 2), (2, 1), (1, 4)],  # Job1
    [(1, 4), (2, 3)],  # Job2
]
""" 
    
machines_count = 1 + max(task[0] for job in jobs_data for task in job)
all_machines = range(machines_count)
# Computes horizon dynamically as the sum of all durations.
horizon = sum(task[1] for job in jobs_data for task in job)

# Create the model.
model = cp_model.CpModel()

# Named tuple to store information about created variables.
task_type = collections.namedtuple("task_type", "start end interval")
# Named tuple to manipulate solution information.
assigned_task_type = collections.namedtuple(
    "assigned_task_type", "start job index duration"
)

# Creates job intervals and add to the corresponding machine lists.
all_tasks = {}
machine_to_intervals = collections.defaultdict(list)

for job_id, job in enumerate(jobs_data):
    for task_id, task in enumerate(job):
        machine, duration = task
        suffix = f"_{job_id}_{task_id}"
        start_var = model.new_int_var(0, horizon, "start" + suffix)
        end_var = model.new_int_var(0, horizon, "end" + suffix)
        interval_var = model.new_interval_var(
            start_var, duration, end_var, "interval" + suffix
        )
        all_tasks[job_id, task_id] = task_type(
            start=start_var, end=end_var, interval=interval_var
        )
        machine_to_intervals[machine].append(interval_var)

# Create and add disjunctive constraints.
for machine in all_machines:
    model.add_no_overlap(machine_to_intervals[machine])

# Precedences inside a job.
for job_id, job in enumerate(jobs_data):
    for task_id in range(len(job) - 1):
        model.add(
            all_tasks[job_id, task_id + 1].start >= all_tasks[job_id, task_id].end
        )

# Makespan objective.
obj_var = model.new_int_var(0, horizon, "makespan")
model.add_max_equality(
    obj_var,
    [all_tasks[job_id, len(job) - 1].end for job_id, job in enumerate(jobs_data)],
)
model.minimize(obj_var)

# Creates the solver and solve.
solver = cp_model.CpSolver()
status = solver.solve(model)

if status == cp_model.OPTIMAL or status == cp_model.FEASIBLE:
    print("Solution:")
    # Create one list of assigned tasks per machine.
    assigned_jobs = collections.defaultdict(list)
    for job_id, job in enumerate(jobs_data):
        for task_id, task in enumerate(job):
            machine = task[0]
            assigned_jobs[machine].append(
                assigned_task_type(
                    start=solver.value(all_tasks[job_id, task_id].start),
                    job=job_id,
                    index=task_id,
                    duration=task[1],
                )
            )

    # Create per machine output lines.
            
    df1 = pd.DataFrame(0, index=range(ope_number), columns=['MC_No', 'Ope_No', 'JB_No', 'ST_T', 'ED_T'])
    i = 0

    output = ""
    for machine in all_machines:
        # Sort by starting time.
        assigned_jobs[machine].sort()
        sol_line_tasks = "Machine " + str(machine) + ": "
        sol_line = "           "
         
        ope_num = 0

        for assigned_task in assigned_jobs[machine]:
              
            df1['MC_No'][i] =  machine

            name = f"job_{assigned_task.job}_task_{assigned_task.index}"
            # add spaces to output to align columns.
                
            df1['Ope_No'][i] = ope_num
            ope_num += 1
             
            df1['JB_No'][i] = assigned_task.job

            sol_line_tasks += f"{name:15}"
            start = assigned_task.start
            duration = assigned_task.duration
            sol_tmp = f"[{start},{start + duration}]"
            # add spaces to output to align columns.
                
            df1['ST_T'][i] = start
            df1['ED_T'][i] = start + duration
            i += 1

            sol_line += f"{sol_tmp:15}"

        sol_line += "\n"
        sol_line_tasks += "\n"
        output += sol_line_tasks
        output += sol_line

    #print(df1)

    # Finally print the solution found.
    print(f"Optimal Schedule Length: {solver.objective_value}")
    print(output)
else:
    print("No solution found.")

# Statistics.
print("\nStatistics")
print(f"  - conflicts: {solver.num_conflicts}")
print(f"  - branches : {solver.num_branches}")
print(f"  - wall time: {solver.wall_time}s")


Solution:
Optimal Schedule Length: 930.0
Machine 0: 
           
Machine 1: job_1_task_0   job_8_task_0   job_0_task_0   job_6_task_1   job_3_task_2   job_4_task_1   job_9_task_1   job_7_task_1   job_5_task_6   job_2_task_1   
           [0,43]         [43,119]       [119,148]      [148,185]      [185,256]      [256,262]      [262,275]      [275,361]      [361,408]      [408,493]      
Machine 2: job_3_task_0   job_5_task_1   job_6_task_0   job_9_task_0   job_8_task_1   job_4_task_2   job_2_task_0   job_7_task_2   job_0_task_1   job_1_task_5   
           [0,81]         [84,86]        [86,132]       [132,217]      [217,286]      [286,308]      [311,402]      [402,448]      [448,526]      [646,674]      
Machine 3: job_5_task_0   job_3_task_1   job_4_task_0   job_7_task_0   job_1_task_1   job_6_task_3   job_9_task_2   job_8_task_4   job_0_task_2   job_2_task_3   
           [0,84]         [84,179]       [179,193]      [193,224]      [224,314]      [314,327]      [327,388]      [421,506]

C:\Users\RJC238\AppData\Local\Temp\ipykernel_17144\1147106472.py:101: FutureWarning:

ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


C:\Users\RJC238\AppData\Local\Temp\ipykernel_17144\1147106472.py:106: FutureWarning:

ChainedAssignmentError: behaviour will 

In [11]:
import plotly.express as px
import plotly.io as pio
import pandas as pd
import numpy as np
import datetime
from datetime import timedelta
import matplotlib.pyplot as plt

#d1 = datetime.date(2024, 3, 31)

d1 = datetime.date(2024, 4, 1)

#d1 = datetime.timedelta()

#print(df1)

#print(d1)
date_st = []
date_ed = []
for i in range(len(df1)):
    d_st = df1['ST_T'].iloc[i]
    d_ed = df1['ED_T'].iloc[i]
    #print(d_st)
    date_st.append(d1 + datetime.timedelta(days=int(d_st)))
    #df1['ST_T'].iloc[i] = date_st
    date_ed.append(d1 + datetime.timedelta(days=int(d_ed)))
    #df1['ED_T'].iloc[i] = date_ed


data = []
for i in range(len(df1)):
    data.append(dict(Task=df1['MC_No'].iloc[i], Start=date_st[i], Finish=date_ed[i], Resource=df1['JB_No'].iloc[i]))
    
df = pd.DataFrame(data)

fig = px.timeline(df, x_start="Start", x_end="Finish", y="Task", color="Resource", title='機械別ガントチャート')

#fig.update_layout(plot_bgcolor="black")
#fig.update_xaxes(linecolor='black', gridcolor='gray',mirror=True)
#fig.update_yaxes(linecolor='black', gridcolor='gray',mirror=True)

fig.update_yaxes(autorange="reversed") # otherwise tasks are listed from the bottom up  

#plt.show()


In [12]:
import plotly.express as px
import plotly.io as pio
import pandas as pd
import numpy as np
import datetime
from datetime import timedelta
import matplotlib.pyplot as plt


#d1 = datetime.date(2024, 3, 31)

d1 = datetime.date(2024, 4, 1)

#d1 = datetime.timedelta()

#print(df1)

#print(d1)
date_st = []
date_ed = []
for i in range(len(df1)):
    d_st = df1['ST_T'].iloc[i]
    d_ed = df1['ED_T'].iloc[i]
    #print(d_st)
    date_st.append(d1 + datetime.timedelta(days=int(d_st)))
    #df1['ST_T'].iloc[i] = date_st
    date_ed.append(d1 + datetime.timedelta(days=int(d_ed)))
    #df1['ED_T'].iloc[i] = date_ed


data = []
for i in range(len(df1)):
    data.append(dict(Task=df1['JB_No'].iloc[i], Start=date_st[i], Finish=date_ed[i], Resource=df1['MC_No'].iloc[i]))
    
df = pd.DataFrame(data)


fig = px.timeline(df, x_start="Start", x_end="Finish", y="Task", color="Resource", title='ジョブ別ガントチャート')
fig.update_yaxes(autorange="reversed") # otherwise tasks are listed from the bottom up  
